# Machine Learning–Based Detection of Malicious Network Traffic

Final project notebook for the Machine Learning Cybersecurity Project. This notebook reproduces the preprocessing, Random Forest training, threshold evaluation, confusion matrices, feature importance, and demonstration predictions described in the project documentation.

In [ ]:
# Run this cell in Google Colab if packages are missing
# !pip install -q pandas numpy matplotlib scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from urllib.parse import urlparse

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score, classification_report)
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

RANDOM_STATE = 42
TEST_SIZE = 0.20
F2_THRESHOLD = 0.2286

In [ ]:
# Upload cybersecurity.csv when using Google Colab
try:
    from google.colab import files
    uploaded = files.upload()
    CSV_PATH = next(iter(uploaded.keys()))
except Exception:
    CSV_PATH = 'cybersecurity.csv'

df_raw = pd.read_csv(CSV_PATH)
df_raw.head()

In [ ]:
REQUIRED_COLUMNS = ['timestamp','src_ip','dst_ip','src_port','dst_port','protocol','bytes_sent','bytes_received','user_agent','url','is_internal_traffic','label','attack_type']
missing_cols = [c for c in REQUIRED_COLUMNS if c not in df_raw.columns]
assert not missing_cols, f'Missing required columns: {missing_cols}'

print('Shape:', df_raw.shape)
print('Exact duplicates:', df_raw.duplicated().sum())
print('Missing values:')
print(df_raw.isna().sum())
print('Class counts:')
print(df_raw['label'].value_counts())

In [ ]:
def clean_url_path(value):
    if pd.isna(value) or str(value).strip() == '':
        return '(missing)'
    try:
        parsed = urlparse(str(value))
        return (parsed.path or '(root)').lower()
    except Exception:
        return '(invalid)'

def prepare_features(df):
    out = df.copy()
    out['label'] = pd.to_numeric(out['label'], errors='coerce').astype('Int64')
    out = out[out['label'].isin([0, 1])].copy()
    out['timestamp'] = pd.to_datetime(out['timestamp'], errors='coerce')
    out['hour'] = out['timestamp'].dt.hour
    out['day_of_week'] = out['timestamp'].dt.dayofweek
    out['url_present'] = (~out['url'].isna() & (out['url'].astype(str).str.strip() != '')).astype(int)
    out['url_path'] = out['url'].apply(clean_url_path)
    out['log_bytes_sent'] = np.log1p(pd.to_numeric(out['bytes_sent'], errors='coerce').clip(lower=0))
    out['log_bytes_received'] = np.log1p(pd.to_numeric(out['bytes_received'], errors='coerce').clip(lower=0))
    out['is_internal_traffic'] = out['is_internal_traffic'].astype(str).str.lower().map({'true':1,'false':0,'1':1,'0':0,'yes':1,'no':0})
    out['src_port'] = pd.to_numeric(out['src_port'], errors='coerce')
    out['dst_port'] = pd.to_numeric(out['dst_port'], errors='coerce')
    return out

df = prepare_features(df_raw)
print(df[['label','hour','day_of_week','url_present','url_path','log_bytes_sent','log_bytes_received']].head())

In [ ]:
NUMERIC_FEATURES = ['src_port','dst_port','log_bytes_sent','log_bytes_received','is_internal_traffic','hour','day_of_week','url_present']
CATEGORICAL_FEATURES = ['protocol','user_agent','url_path']
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X = df[FEATURES]
y = df['label'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
print('Train size:', len(X_train), 'Test size:', len(X_test))
print('Test class counts:')
print(y_test.value_counts())

In [ ]:
numeric_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', RobustScaler())])
categorical_pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer([('num', numeric_pipeline, NUMERIC_FEATURES), ('cat', categorical_pipeline, CATEGORICAL_FEATURES)])
rf = RandomForestClassifier(n_estimators=150, min_samples_leaf=3, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1)
pipe = Pipeline([('preprocess', preprocessor), ('model', rf)])

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
oof_proba = cross_val_predict(pipe, X_train, y_train, cv=cv, method='predict_proba')[:, 1]
print('Training CV PR-AUC:', average_precision_score(y_train, oof_proba))
print('Training CV ROC-AUC:', roc_auc_score(y_train, oof_proba))

In [ ]:
pipe.fit(X_train, y_train)
test_proba = pipe.predict_proba(X_test)[:, 1]
print('Held-out PR-AUC:', average_precision_score(y_test, test_proba))
print('Held-out ROC-AUC:', roc_auc_score(y_test, test_proba))

def metrics_at_threshold(threshold):
    pred = (test_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0,1]).ravel()
    return {
        'threshold': threshold,
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
        'false_positive_rate': fp / (fp + tn),
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp
    }

results = pd.DataFrame([metrics_at_threshold(0.50), metrics_at_threshold(F2_THRESHOLD)])
results

In [ ]:
for threshold in [0.50, F2_THRESHOLD]:
    pred = (test_proba >= threshold).astype(int)
    cm = confusion_matrix(y_test, pred, labels=[0,1])
    fig, ax = plt.subplots(figsize=(4,3.5))
    ax.imshow(cm, cmap='Blues')
    ax.set_title(f'Confusion Matrix @ {threshold:.3f}')
    ax.set_xticks([0,1], ['Benign','Attack'])
    ax.set_yticks([0,1], ['Benign','Attack'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i,j]), ha='center', va='center', fontweight='bold')
    plt.show()
    print(classification_report(y_test, pred, zero_division=0))

In [ ]:
# Group impurity-based importances back to original feature groups.
pre = pipe.named_steps['preprocess']
model = pipe.named_steps['model']
encoded_names = list(NUMERIC_FEATURES) + list(pre.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(CATEGORICAL_FEATURES))
imp = pd.DataFrame({'encoded_feature': encoded_names, 'importance': model.feature_importances_})

def group_name(name):
    for cat in CATEGORICAL_FEATURES:
        if name.startswith(cat + '_'):
            return cat
    if name == 'log_bytes_sent':
        return 'bytes_sent'
    if name == 'log_bytes_received':
        return 'bytes_received'
    return name

imp['feature_group'] = imp['encoded_feature'].apply(group_name)
grouped_importance = imp.groupby('feature_group')['importance'].sum().sort_values(ascending=False)
print((grouped_importance * 100).round(2))

grouped_importance.head(10).sort_values().plot(kind='barh', figsize=(7,4), title='Top Grouped Feature Importances')
plt.xlabel('Importance')
plt.show()

In [ ]:
# Demonstration: manual prediction using the trained pipeline
sample_event = pd.DataFrame([{
    'timestamp': '2025-10-01 12:00:00',
    'src_ip': '0.0.0.0',
    'dst_ip': '0.0.0.0',
    'src_port': 443,
    'dst_port': 80,
    'protocol': 'TCP',
    'bytes_sent': 1000,
    'bytes_received': 5000,
    'user_agent': 'Mozilla/5.0',
    'url': 'https://example.com/login',
    'is_internal_traffic': True,
    'label': 0,
    'attack_type': 'unknown'
}])

sample_prepared = prepare_features(sample_event)[FEATURES]
probability = pipe.predict_proba(sample_prepared)[:, 1][0]
print(f'Predicted attack probability: {probability:.2%}')
print('Prediction @ 0.50:', 'Attack' if probability >= 0.50 else 'Benign')
print(f'Prediction @ F2 threshold {F2_THRESHOLD:.3f}:', 'Attack' if probability >= F2_THRESHOLD else 'Benign')